# AI Agent-Based RAG Pipeline for Car Price Prediction

This notebook implements a RAG (Retrieval-Augmented Generation) pipeline using AI agents to:
1. Understand natural language queries about cars
2. Retrieve relevant cars from the CSV database
3. Use the ML model to predict prices
4. Generate comprehensive responses using AI agents


## 1. Install Required Packages


In [28]:
%pip install 'langchain<0.1.0'  langgraph langchain langchain-openai langchain-community langchain-experimental pandas numpy scikit-learn catboost sentence-transformers faiss-cpu python-dotenv


Note: you may need to restart the kernel to use updated packages.


## 2. Load Environment Variables


In [3]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Get the directory where this notebook is located
notebook_dir = Path.cwd()
env_file = notebook_dir / ".env"

# Check if .env file exists
if not env_file.exists():
    print(f"⚠️  Warning: .env file not found at {env_file}")
    print("\nPlease create a .env file with your OpenAI API key:")
    print("OPENAI_API_KEY=sk-your-actual-api-key-here\n")
    raise FileNotFoundError(f".env file not found at {env_file}")
else:
    print(f"✅ Found .env file at {env_file}")
    
    

# Load environment variables from .env file (override=True to ensure latest values)
load_dotenv(dotenv_path=env_file, override=True)

# Get API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# If key is too long or has issues, manually parse the file
if OPENAI_API_KEY and len(OPENAI_API_KEY) > 100:
    print(f"\n⚠️  Key seems too long ({len(OPENAI_API_KEY)} chars). Manually parsing .env file...")
    with open(env_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('OPENAI_API_KEY'):
                # Handle both "KEY=value" and "KEY = value" formats
                if '=' in line:
                    # Split on = and take everything after it
                    potential_key = line.split('=', 1)[1].strip()
                    # Extract just the part that looks like an API key (starts with sk-)
                    if 'sk-' in potential_key:
                        # Find where sk- starts
                        sk_index = potential_key.find('sk-')
                        OPENAI_API_KEY = potential_key[sk_index:]
                        # Take only up to the first space or newline after sk-
                        if ' ' in OPENAI_API_KEY:
                            OPENAI_API_KEY = OPENAI_API_KEY.split()[0]
                        print(f"✅ Extracted key: {OPENAI_API_KEY[:10]}...{OPENAI_API_KEY[-6:]} ({len(OPENAI_API_KEY)} chars)")
                    break

# Always strip whitespace
if OPENAI_API_KEY:
    OPENAI_API_KEY = OPENAI_API_KEY.strip()



✅ Found .env file at /Users/supimraid/DSA/DSA Internal Grp $/RAG Pipeline/.env

⚠️  Key seems too long (164 chars). Manually parsing .env file...
✅ Extracted key: sk-proj-5K...tiNNAA (164 chars)


## 3. Initialize Components


In [31]:
from typing_extensions import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolExecutor
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, ToolMessage

# 1. Define agent state
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# 2. Define a tool
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# MUST be a list, not dict
tools = [add_numbers]

# 3. Load LLM and bind tools
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

# 4. LLM node wrapper
def llm_node(state: AgentState):
    msg = llm_with_tools.invoke(state["messages"])
    return {"messages": [msg]}

# 5. Tool node using ToolExecutor (compatible with langgraph 0.0.24)
tool_executor = ToolExecutor(tools)

def tool_node(state: AgentState):
    """Execute tools based on the last message"""
    messages = state["messages"]
    last_message = messages[-1]
    
    # Check if the last message has tool calls
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        results = []
        for tool_call in last_message.tool_calls:
            tool_result = tool_executor.invoke(tool_call)
            results.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call.get("id") if isinstance(tool_call, dict) else getattr(tool_call, "id", None)
                )
            )
        return {"messages": results}
    return {"messages": []}

# 6. Build graph
workflow = StateGraph(AgentState)

workflow.add_node("llm", llm_node)
workflow.add_node("tools", tool_node)

workflow.set_entry_point("llm")

# Add conditional edge: if LLM wants to call tools, go to tools, otherwise end
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    # If there are tool calls, go to tools node
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    # Otherwise, end
    return "__end__"

workflow.add_conditional_edges("llm", should_continue)
# After tools, always go back to LLM
workflow.add_edge("tools", "llm")

app = workflow.compile()

print("✅ Agent created successfully")


✅ Agent created successfully


In [33]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import Tool
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
import warnings
warnings.filterwarnings("ignore")

# Try different import paths for AgentExecutor (LangChain version compatibility)
# try:
#     from langchain.agents import AgentExecutor, create_openai_tools_agent
# except ImportError:
#     try:
#         from langchain.agents.agent_executor import AgentExecutor
#         from langchain.agents import create_openai_tools_agent
#     except ImportError:
#         from langchain_core.agents import AgentExecutor
#         from langchain.agents import create_openai_tools_agent

print("✅ All imports successful")


✅ All imports successful


## 4. Load Data and Model


In [34]:
# Load the trained model
print("Loading ML model...")
price_model = CatBoostRegressor()
price_model.load_model('/Users/supimraid/Downloads/catboost_info.cbm')
print("✅ Model loaded")

# Load and preprocess data
print("Loading CSV data...")
df = pd.read_csv("car_prices_extended_eda.csv", parse_dates=False)

# Drop columns that were dropped in training
drop_cols = ["vin", "saledate", "price_diff"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Apply log transformation to skewed columns (as in training)
skewed_cols = ["odometer", "car_age"]
for col in skewed_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col])

print(f"✅ Data loaded: {len(df)} cars")


Loading ML model...
✅ Model loaded
Loading CSV data...
✅ Data loaded: 472325 cars


## 5. Create Vector Store for Semantic Search


In [18]:
def create_car_description(row):
    """Create a text description of a car for embedding"""
    desc_parts = []
    
    if 'year' in row and pd.notna(row['year']):
        desc_parts.append(f"{int(row['year'])}")
    if 'make' in row and pd.notna(row['make']):
        desc_parts.append(row['make'])
    if 'model' in row and pd.notna(row['model']):
        desc_parts.append(row['model'])
    if 'trim' in row and pd.notna(row['trim']):
        desc_parts.append(row['trim'])
    if 'body' in row and pd.notna(row['body']):
        desc_parts.append(f"{row['body']} body")
    if 'transmission' in row and pd.notna(row['transmission']):
        desc_parts.append(f"{row['transmission']} transmission")
    if 'color' in row and pd.notna(row['color']):
        desc_parts.append(f"{row['color']} color")
    if 'condition' in row and pd.notna(row['condition']):
        desc_parts.append(f"condition score {row['condition']}")
    if 'odometer' in row and pd.notna(row['odometer']):
        odometer_original = np.expm1(row['odometer'])
        desc_parts.append(f"{odometer_original:.0f} miles")
    if 'state' in row and pd.notna(row['state']):
        desc_parts.append(f"located in {row['state']}")
    
    return " ".join(desc_parts)

print("Creating car descriptions...")
car_descriptions = []
for idx, row in df.iterrows():
    desc = create_car_description(row)
    car_descriptions.append(desc)

print("Creating vector store with OpenAI embeddings...")
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Create documents for LangChain
documents = []
for idx, (desc, row) in enumerate(zip(car_descriptions, df.itertuples())):
    # Create metadata with all car information
    metadata = row._asdict()
    # Convert to dict and remove Index
    metadata = {k: v for k, v in metadata.items() if k != 'Index'}
    doc = Document(page_content=desc, metadata=metadata)
    documents.append(doc)

# Create FAISS vector store
vector_store = FAISS.from_documents(documents, embeddings)
print(f"✅ Vector store created with {len(documents)} cars")


Creating car descriptions...
Creating vector store with OpenAI embeddings...
✅ Vector store created with 472325 cars


## 6. Create Tools for AI Agent


In [36]:
def search_similar_cars(query: str, top_k: int = 5) -> str:
    """
    Search for similar cars based on a natural language query.
    
    Args:
        query: Natural language description of the car
        top_k: Number of results to return
    
    Returns:
        A formatted string with car information
    """
    results = vector_store.similarity_search(query, k=top_k)
    
    output = []
    for i, doc in enumerate(results, 1):
        car_info = doc.metadata
        output.append(f"\nCar {i}:")
        output.append(f"  Description: {doc.page_content}")
        output.append(f"  Year: {car_info.get('year', 'N/A')}")
        output.append(f"  Make: {car_info.get('make', 'N/A')}")
        output.append(f"  Model: {car_info.get('model', 'N/A')}")
        output.append(f"  Trim: {car_info.get('trim', 'N/A')}")
        output.append(f"  Body: {car_info.get('body', 'N/A')}")
        output.append(f"  Transmission: {car_info.get('transmission', 'N/A')}")
        output.append(f"  Color: {car_info.get('color', 'N/A')}")
        output.append(f"  Condition: {car_info.get('condition', 'N/A')}")
        if 'odometer' in car_info:
            odometer_original = np.expm1(car_info['odometer'])
            output.append(f"  Odometer: {odometer_original:.0f} miles")
        output.append(f"  State: {car_info.get('state', 'N/A')}")
        if 'sellingprice' in car_info:
            output.append(f"  Actual Price: ${car_info.get('sellingprice', 0):,.2f}")
    
    return "\n".join(output)

def predict_car_price(car_features: str) -> str:
    """
    Predict the price of a car based on its features.
    The input should be a description of car features that can be matched to the database.
    
    Args:
        car_features: Description of car features (e.g., '2015 BMW 3 Series sedan')
    
    Returns:
        Predicted price and car information
    """
    # First, find the most similar car
    results = vector_store.similarity_search(car_features, k=1)
    
    if not results:
        return "No matching car found in the database."
    
    doc = results[0]
    car_info = doc.metadata
    
    # Prepare features for prediction (same as training)
    feature_cols = [col for col in df.columns if col != 'sellingprice']
    
    features = []
    for col in feature_cols:
        if col in car_info:
            features.append(car_info[col])
        else:
            # Use median if missing
            features.append(df[col].median())
    
    features_array = np.array(features).reshape(1, -1)
    
    # Predict
    prediction = price_model.predict(features_array)[0]
    
    output = []
    output.append(f"Car: {doc.page_content}")
    output.append(f"Predicted Price: ${prediction:,.2f}")
    if 'sellingprice' in car_info:
        actual = car_info['sellingprice']
        error = abs(prediction - actual)
        error_pct = (error / actual) * 100 if actual > 0 else 0
        output.append(f"Actual Price: ${actual:,.2f}")
        output.append(f"Prediction Error: ${error:,.2f} ({error_pct:.1f}%)")
    
    return "\n".join(output)

def get_car_statistics(filters: str = "") -> str:
    """
    Get statistics about cars in the database.
    
    Args:
        filters: Optional filter description (e.g., 'BMW cars' or 'sedans')
    
    Returns:
        Statistics about the cars
    """
    if filters:
        # Try to filter based on description
        results = vector_store.similarity_search(filters, k=min(1000, len(df)))
        filtered_df = df.iloc[[i for i in range(len(results))]]
    else:
        filtered_df = df
    
    if 'sellingprice' in filtered_df.columns:
        prices = filtered_df['sellingprice']
        stats = []
        stats.append(f"Total cars: {len(filtered_df)}")
        stats.append(f"Average price: ${prices.mean():,.2f}")
        stats.append(f"Median price: ${prices.median():,.2f}")
        stats.append(f"Min price: ${prices.min():,.2f}")
        stats.append(f"Max price: ${prices.max():,.2f}")
        
        if 'make' in filtered_df.columns:
            top_makes = filtered_df['make'].value_counts().head(5)
            stats.append(f"\nTop 5 makes:")
            for make, count in top_makes.items():
                stats.append(f"  {make}: {count}")
        
        return "\n".join(stats)
    else:
        return f"Total cars: {len(filtered_df)}"

# Create tools for the agent
tools = [
    Tool(
        name="search_similar_cars",
        func=search_similar_cars,
        description="Search for cars similar to a given description. Use this to find cars matching specific criteria like make, model, year, body type, etc. Input should be a natural language description."
    ),
    Tool(
        name="predict_car_price",
        func=predict_car_price,
        description="Predict the price of a car based on its features. Input should be a description of the car (e.g., '2015 BMW 3 Series sedan'). This will find the most similar car and predict its price using the ML model."
    ),
    Tool(
        name="get_car_statistics",
        func=get_car_statistics,
        description="Get statistics about cars in the database. Can optionally filter by description (e.g., 'BMW cars' or 'sedans'). Returns price statistics and top makes."
    )
]

print("✅ Tools created")


✅ Tools created


## 7. Initialize AI Agent


In [38]:
from typing_extensions import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolExecutor
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, AnyMessage

# Define agent state (compatible with langgraph 0.0.24)
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    input: str
    agent_scratchpad: list[AnyMessage]

# 1. LLM (tools are defined in Cell 16, so we bind them here)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)
llm_with_tools = llm.bind_tools(tools)

# 2. Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI assistant specialized in car price prediction and analysis.
You have access to a database of cars and a trained ML model for price prediction.

Your capabilities:
1. Search for cars matching specific criteria
2. Predict car prices using the ML model
3. Provide statistics about the car database

When answering questions:
- Use the search_similar_cars tool to find relevant cars
- Use predict_car_price to get price predictions
- Use get_car_statistics for aggregate information
- Provide clear, comprehensive answers with relevant details
- Always include price predictions when discussing specific cars

Be conversational and helpful. If a user asks about a car, search for similar cars and provide predictions.
"""),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 3. LLM node
def llm_node(state: AgentState):
    messages = state.get("messages", [])
    user_input = state.get("input", "")
    
    # If no messages yet, start with user input
    if not messages and user_input:
        messages = [HumanMessage(content=user_input)]
    
    # Call LLM with tools bound
    result = llm_with_tools.invoke(messages)
    # result is an AIMessage
    
    return {
        "messages": [result]
    }

# 4. Tools node using ToolExecutor (compatible with langgraph 0.0.24)
tool_executor = ToolExecutor(tools)

def tool_node_func(state: AgentState):
    """Execute tools based on the last message"""
    messages = state.get("messages", [])
    if not messages:
        return {"messages": []}
    
    last_message = messages[-1]
    
    # Check if the last message has tool calls
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        results = []
        for tool_call in last_message.tool_calls:
            tool_result = tool_executor.invoke(tool_call)
            results.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call.get("id") if isinstance(tool_call, dict) else getattr(tool_call, "id", None)
                )
            )
        return {"messages": results}
    return {"messages": []}

# 5. Build workflow
workflow = StateGraph(AgentState)
workflow.add_node("llm", llm_node)
workflow.add_node("tools", tool_node_func)
workflow.set_entry_point("llm")

# Add conditional edge: if LLM wants to call tools, go to tools, otherwise end
def should_continue(state: AgentState):
    messages = state.get("messages", [])
    if not messages:
        return "__end__"
    last_message = messages[-1]
    # If there are tool calls, go to tools node
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    # Otherwise, end
    return "__end__"

workflow.add_conditional_edges("llm", should_continue)
# After tools, always go back to LLM
workflow.add_edge("tools", "llm")

app = workflow.compile()
print("✅ Agent created successfully using LangGraph")


✅ Agent created successfully using LangGraph


In [46]:
# FIX: Rebuild workflow with corrected llm_node function
# The issue is that llm_node wasn't including system message and all previous messages

# Redefine llm_node with proper message handling
def llm_node_fixed(state):
    # Safely access state as dict (TypedDict)
    if isinstance(state, dict):
        messages = state.get("messages", [])
        user_input = state.get("input", "")
    else:
        # Handle case where state might be accessed differently
        try:
            messages = getattr(state, "messages", [])
            user_input = getattr(state, "input", "")
        except:
            messages = []
            user_input = ""
    
    # Build system message
    system_msg = SystemMessage(content="You are a helpful AI assistant specialized in car price prediction and analysis. "
                                      "You have access to a database of cars and a trained ML model for price prediction. "
                                      "Your capabilities: 1) Search for cars matching specific criteria, "
                                      "2) Predict car prices using the ML model, 3) Provide statistics about the car database. "
                                      "When answering questions, use the available tools to search for cars and predict prices. "
                                      "After using tools, always provide a clear, comprehensive answer summarizing the results. "
                                      "Provide clear, comprehensive answers with relevant details.")
    
    # If no messages yet, start with system message and user input
    if not messages:
        messages = [system_msg, HumanMessage(content=user_input)]
    else:
        # Messages exist (includes tool results via add_messages), ensure system message is first
        if not isinstance(messages[0], SystemMessage):
            messages = [system_msg] + messages
    
    # Call LLM with tools bound - this will see all previous messages including tool results
    result = llm_with_tools.invoke(messages)
    
    return {
        "messages": [result]
    }

# Create a wrapper for tool_node_func to handle state access safely
def tool_node_func_safe(state):
    # Safely access state as dict (TypedDict)
    if isinstance(state, dict):
        messages = state.get("messages", [])
    else:
        try:
            messages = getattr(state, "messages", [])
        except:
            messages = []
    
    if not messages:
        return {"messages": []}
    
    last_message = messages[-1]
    
    # Check if the last message has tool calls
    tool_calls = None
    if hasattr(last_message, 'tool_calls'):
        tool_calls = last_message.tool_calls
    elif isinstance(last_message, dict):
        tool_calls = last_message.get('tool_calls', None)
    
    if tool_calls:
        results = []
        for tool_call in tool_calls:
            tool_result = tool_executor.invoke(tool_call)
            results.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call.get("id") if isinstance(tool_call, dict) else getattr(tool_call, "id", None)
                )
            )
        return {"messages": results}
    return {"messages": []}

# Rebuild the workflow with the fixed functions
workflow_fixed = StateGraph(AgentState)
workflow_fixed.add_node("llm", llm_node_fixed)
workflow_fixed.add_node("tools", tool_node_func_safe)
workflow_fixed.set_entry_point("llm")

# Add conditional edge: if LLM wants to call tools, go to tools, otherwise end
def should_continue(state):
    # Safely access state as dict (TypedDict)
    if isinstance(state, dict):
        messages = state.get("messages", [])
    else:
        # Handle case where state might be accessed differently
        try:
            messages = getattr(state, "messages", [])
        except:
            messages = []
    
    if not messages:
        return "__end__"
    
    last_message = messages[-1]
    
    # Check if the last message has tool calls
    # Handle both dict-like and object-like access
    tool_calls = None
    if hasattr(last_message, 'tool_calls'):
        tool_calls = last_message.tool_calls
    elif isinstance(last_message, dict):
        tool_calls = last_message.get('tool_calls', None)
    
    if tool_calls:
        return "tools"
    # Otherwise, end
    return "__end__"

workflow_fixed.add_conditional_edges("llm", should_continue)
# After tools, always go back to LLM
workflow_fixed.add_edge("tools", "llm")

# Replace the app with the fixed version
app = workflow_fixed.compile()
print("✅ Workflow rebuilt with fixed llm_node function")


✅ Workflow rebuilt with fixed llm_node function


## 8. Test the RAG Pipeline with AI Agent


In [52]:
query = "Find me 2015 BMW sedans with low mileage and predict their prices"

response = app.invoke({
    "input": query,
    "messages": [],  # Initialize messages list
    "agent_scratchpad": []  # must be a list
})

# Access content as attribute, not dictionary key (AIMessage objects use .content, not ["content"])
last_message = response["messages"][-1]
print(last_message.content)


## 9. Interactive Query Interface


In [51]:
# Debug: Check what messages are being generated
test_query = "Find me a red sports car from 2014-2015 and predict its price"
test_result = app.invoke({
    "input": test_query,
    "messages": [],
    "agent_scratchpad": []
})

print("Total messages:", len(test_result.get("messages", [])))
print("\nAll messages:")
for i, msg in enumerate(test_result.get("messages", [])):
    msg_type = type(msg).__name__
    content_preview = str(msg.content)[:100] if hasattr(msg, 'content') else str(msg)[:100]
    print(f"\nMessage {i} ({msg_type}):")
    print(f"  Content: {content_preview}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  Tool calls: {len(msg.tool_calls)}")


Total messages: 1

All messages:

Message 0 (AIMessage):
  Content: 


In [ ]:
def ask_agent(question: str) -> str:
    """
    Ask the AI agent a question about cars.
    
    Args:
        question: Your question about cars
    
    Returns:
        Agent's response
    """
    # Use the LangGraph app (created in Cell 18)
    result = app.invoke({
        "input": question,
        "messages": [],
        "agent_scratchpad": []
    })
    
    # Extract the content from the last message
    last_message = result["messages"][-1]
    return last_message.content

# Example usage
question = "Find me a red sports car from 2014-2015 and predict its price"
answer = ask_agent(question)
print(f"Question: {question}\n")
print(f"Answer:\n{answer}")


Question: Find me a red sports car from 2014-2015 and predict its price

Answer:
Empty response.


## 10. Save Vector Store (Optional - for faster loading next time)


In [49]:
# Save vector store for future use
vector_store.save_local("car_vector_store")
print("✅ Vector store saved to 'car_vector_store' directory")


✅ Vector store saved to 'car_vector_store' directory


## 11. Load Saved Vector Store (Optional - use this instead of creating new one)


In [ ]:
# Load FAISS Vector Store and PKL Files for AI Agent
import pickle
import os
from pathlib import Path

# 1. Load FAISS Vector Store
print("Loading FAISS vector store...")
try:
    # Ensure embeddings are available (recreate if needed)
    if 'embeddings' not in globals():
        from langchain_openai import OpenAIEmbeddings
        embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
        print("✅ Created embeddings instance")
    
    # Load the vector store
    vector_store_path = "car_vector_store"
    if os.path.exists(vector_store_path):
        vector_store = FAISS.load_local(
            vector_store_path, 
            embeddings, 
            allow_dangerous_deserialization=True
        )
        print(f"✅ Vector store loaded from '{vector_store_path}' directory")
        print(f"   Total documents: {len(vector_store.docstore._dict)}")
    else:
        print(f"⚠️  Vector store not found at '{vector_store_path}'")
        vector_store = None
except Exception as e:
    print(f"❌ Error loading vector store: {e}")
    vector_store = None

# 2. Load PKL Files (if any)
print("\nLoading PKL files...")
pkl_files = {}

# Common pkl file paths to check
pkl_paths = [
    "car_vector_store/index.pkl",  # FAISS index metadata (already loaded with FAISS)
    "price_model.pkl",  # ML model if saved as pkl
    "model.pkl",
    "data.pkl",
    "preprocessor.pkl",
]

for pkl_path in pkl_paths:
    if os.path.exists(pkl_path):
        try:
            with open(pkl_path, 'rb') as f:
                pkl_data = pickle.load(f)
                pkl_name = Path(pkl_path).stem
                pkl_files[pkl_name] = pkl_data
                print(f"✅ Loaded {pkl_path}")
        except Exception as e:
            print(f"⚠️  Could not load {pkl_path}: {e}")

# 3. Load ML Model (CatBoost) if available
print("\nLoading ML model...")
try:
    from catboost import CatBoostRegressor
    
    # Check for .cbm file (CatBoost model format)
    model_paths = [
        "catboost_info.cbm",
        "/Users/supimraid/Downloads/catboost_info.cbm",
        "price_model.cbm",
        "model.cbm"
    ]
    
    price_model = None
    for model_path in model_paths:
        if os.path.exists(model_path):
            price_model = CatBoostRegressor()
            price_model.load_model(model_path)
            print(f"✅ ML model loaded from '{model_path}'")
            break
    
    if price_model is None:
        print("⚠️  ML model (.cbm file) not found. Please ensure the model file exists.")
        
except Exception as e:
    print(f"⚠️  Error loading ML model: {e}")
    price_model = None

# Summary
print("\n" + "="*60)
print("LOADING SUMMARY")
print("="*60)
print(f"Vector Store: {'✅ Loaded' if vector_store else '❌ Not loaded'}")
print(f"ML Model: {'✅ Loaded' if price_model else '❌ Not loaded'}")
print(f"PKL Files: {len(pkl_files)} file(s) loaded")
if pkl_files:
    for name in pkl_files.keys():
        print(f"  - {name}")
print("="*60)

# Make variables available globally for the AI agent
if vector_store:
    print("\n✅ Vector store is ready for use in AI agent tools")
if price_model:
    print("✅ ML model is ready for use in AI agent tools")


Loading FAISS vector store...
❌ Error loading vector store: '__fields_set__'

Loading PKL files...
⚠️  Could not load car_vector_store/index.pkl: '__fields_set__'

Loading ML model...
✅ ML model loaded from '/Users/supimraid/Downloads/catboost_info.cbm'

LOADING SUMMARY
Vector Store: ❌ Not loaded
ML Model: ✅ Loaded
PKL Files: 0 file(s) loaded
✅ ML model is ready for use in AI agent tools
